In [25]:
import re
import csv
from collections import Counter

# Sample log file path
path = 'sample.log'
FAILED_LOGIN_THRESHOLD = 10  # Configurable threshold for failed login attempts

def parse_log_file(path):
    # Initialize counters for different metrics
    ip_counter = Counter()
    endpoint_counter = Counter()
    failed_login_counts = {}

    # Regular expressions for extracting data
    failed_login_pattern = r'POST /login.* (401) .*("Invalid credentials"|\b401\b)'
    endpoint_pattern = r'\"(GET|POST) (\S+)'

    # Read the log file
    with open(path, 'r') as file:
        log_lines = file.readlines()

    for line in log_lines:
        # Count IP addresses (for requests per IP)
        ip_match = re.match(r'(\d+\.\d+\.\d+\.\d+)', line)
        if ip_match:
            ip_address = ip_match.group(1)
            ip_counter[ip_address] += 1
        
        # Count endpoints (most accessed endpoint)
        endpoint_match = re.search(endpoint_pattern, line)
        if endpoint_match:
            endpoint = endpoint_match.group(2)
            endpoint_counter[endpoint] += 1

        # Detect failed login attempts (suspicious activity)
        match = re.search(failed_login_pattern, line)
        if match:
            ip_address = ip_match.group(1)
            if ip_address in failed_login_counts:
                failed_login_counts[ip_address] += 1
            else:
                failed_login_counts[ip_address] = 1

    return ip_counter, endpoint_counter, failed_login_counts

def detect_suspicious_activity(failed_login_counts, threshold=FAILED_LOGIN_THRESHOLD):
    # Filter out IPs with failed login attempts above the threshold
    suspicious_activity = {ip: count for ip, count in failed_login_counts.items() if count > threshold}
    return suspicious_activity

def output_results(ip_counter, endpoint_counter, suspicious_activity):
    # Sort IP addresses by request count in descending order
    sorted_ip_counts = sorted(ip_counter.items(), key=lambda x: x[1], reverse=True)

    # Display results in the terminal
    print("Requests per IP:")
    print(f"{'IP Address':<20} {'Request Count'}")
    for ip, count in sorted_ip_counts:
        print(f"{ip:<20} {count}")

    print("\nMost Frequently Accessed Endpoint:")
    most_accessed_endpoint = endpoint_counter.most_common(1)[0]
    print(f"{most_accessed_endpoint[0]} (Accessed {most_accessed_endpoint[1]} times)")

    print("\nSuspicious Activity Detected:")
    print(f"{'IP Address':<20} {'Failed Login Attempts'}")
    for ip, count in suspicious_activity.items():
        print(f"{ip:<20} {count}")

    # Save results to a CSV file
    with open('log_analysis_results.csv', mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # Write Requests per IP (sorted by request count)
        writer.writerow(['IP Address', 'Request Count'])
        for ip, count in sorted_ip_counts:
            writer.writerow([ip, count])

        # Write Most Accessed Endpoint
        writer.writerow(['Endpoint', 'Access Count'])
        endpoint = endpoint_counter.most_common(1)[0]
        writer.writerow([endpoint[0], endpoint[1]])

        # Write Suspicious Activity
        writer.writerow(['IP Address', 'Failed Login Count'])
        for ip, count in suspicious_activity.items():
            writer.writerow([ip, count])

def main():
    # Parse the log file and extract the required data
    ip_counter, endpoint_counter, failed_login_counts = parse_log_file(log_file_path)

    # Detect suspicious activity (brute force login attempts)
    suspicious_activity = detect_suspicious_activity(failed_login_counts)

    # Output results to terminal and save to CSV
    output_results(ip_counter, endpoint_counter, suspicious_activity)

# Execute the main function
if __name__ == '__main__':
    main()


Requests per IP:
IP Address           Request Count
203.0.113.5          8
198.51.100.23        8
192.168.1.1          7
10.0.0.2             6
192.168.1.100        5

Most Frequently Accessed Endpoint:
/login (Accessed 13 times)

Suspicious Activity Detected:
IP Address           Failed Login Attempts
